# Structured credit 3 — Auto ABS & RMBS (debt + residual)

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/instruments/structured_credit_clo_debt.ipynb` and `02_pricing/instruments/structured_credit_clo_equity.ipynb`.

Consumer **ABS** and **RMBS** are backed by **amortizing** collateral (auto loans,
mortgages) that pays *scheduled* principal every month plus **prepayments**. This
notebook prices a senior **debt** tranche and the **residual** for each, using
**PSA** prepayment and **SDA** default ramps, and stresses them on scenario grids.


## Concept

- **Amortizing pools:** unlike CLO bullet loans, auto/mortgage collateral returns
  principal monthly; **prepayment** (CPR / PSA) drives the debt **WAL**.
- **Sequential pay:** the senior **A** note is paid down first; the **residual**
  absorbs first loss and keeps the excess.
- **Standard ramps:** **PSA** (prepayment seasoning ramp, 100% = the SIFMA benchmark)
  and **SDA** (standard default assumption) are first-class curve inputs.
- We price PV / **WAL** per tranche and stress on **CDR × severity** (ABS) and
  **CPR × CDR** (RMBS) grids.

In [ ]:
import json
import pandas as pd
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, ForwardCurve, MarketContext
from finstack_quant.valuations.instruments.fixed_income import StructuredCredit

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
AS_OF = "2024-06-15"
AS_OF_D = date(2024, 6, 15)
print("Imports OK (structured credit).")

In [ ]:
# --- thin builders over the structured-credit serde spec (Rust is canonical) ---
def money(x):
    return {"amount": str(x), "currency": "USD"}

def asset(aid, atype, bal, rate, spread_bps=None, index_id=None,
          maturity="2031-06-15", day_count="Act360"):
    """One pool asset. Floating loans set spread_bps + index_id (rate = fallback)."""
    return {"id": aid, "asset_type": atype, "balance": money(bal), "rate": rate,
            "spread_bps": spread_bps, "index_id": index_id, "maturity": maturity,
            "credit_quality": None, "industry": None, "obligor_id": None,
            "is_defaulted": False, "recovery_amount": None, "purchase_price": None,
            "acquisition_date": None, "day_count": day_count,
            "smm_override": None, "mdr_override": None}

def floating(spread_bp, index_id="USD-SOFR-3M"):
    """Minimal floating tranche coupon (index + spread); other fields take engine defaults."""
    return {"Floating": {"index_id": index_id, "spread_bp": spread_bp,
            "reset_freq": {"count": 3, "unit": "months"}, "dc": "Act360", "calendar_id": "nyse"}}

def fixed(rate):
    return {"Fixed": {"rate": rate}}

def tranche(tid, att, det, seniority, bal, coupon, priority, freq_n=3, maturity="2031-06-15"):
    """One liability tranche. attachment/detachment are 0-100 (percent of structure)."""
    return {"id": tid, "attachment_point": att, "detachment_point": det,
            "behavior_type": "Standard", "seniority": seniority, "rating": None,
            "original_balance": money(bal), "current_balance": money(bal), "target_balance": None,
            "coupon": coupon, "oc_trigger": None, "ic_trigger": None,
            "credit_enhancement": {"subordination": money(0), "overcollateralization": money(0),
                                   "reserve_account": money(0), "excess_spread": 0.0,
                                   "cash_trap_active": False},
            "frequency": {"count": freq_n, "unit": "months"}, "day_count": "Act360",
            "deferred_interest": money(0), "pik_enabled": False, "is_revolving": False,
            "can_reinvest": False, "maturity": maturity, "expected_maturity": None,
            "payment_priority": priority, "attributes": {}}

def pool(pid, deal_type, assets, reinvest_end=None):
    """Collateral pool. reinvest_end (a date string) opens a reinvestment period."""
    rp = None
    if reinvest_end is not None:
        rp = {"end_date": reinvest_end, "is_active": True,
              "criteria": {"max_price": 101.0, "min_yield": 0.0,
                           "maintain_credit_quality": True, "maintain_wal": True,
                           "apply_eligibility_criteria": True}}
    return {"id": pid, "deal_type": deal_type, "base_currency": "USD", "assets": assets,
            "cumulative_defaults": money(0), "cumulative_recoveries": money(0),
            "cumulative_prepayments": money(0), "cumulative_scheduled_amortization": None,
            "reinvestment_period": rp, "collection_account": money(0),
            "reserve_account": money(0), "excess_spread_account": money(0), "rep_lines": None}

def build(did, deal_type, pool_dict, tranches, freq_n, closing, first_pay, maturity,
          reinvest_end=None, prepay=None, default=None, recovery=None, reinvestment_price=None):
    """Assemble + return (StructuredCredit, spec). reinvestment_price is % of par (e.g. 97)."""
    prepay = prepay if prepay is not None else {"cpr": 0.0, "curve": None}
    default = default if default is not None else {"cdr": 0.0, "curve": None}
    recovery = recovery if recovery is not None else {"rate": 0.40, "recovery_lag": 0}
    total = sum(int(t["original_balance"]["amount"]) for t in tranches)
    spec = {"id": did, "deal_type": deal_type, "pool": pool_dict,
            "tranches": {"tranches": tranches, "total_size": money(total)},
            "closing_date": closing, "first_payment_date": first_pay,
            "reinvestment_end_date": reinvest_end, "maturity": maturity,
            "frequency": {"count": freq_n, "unit": "months"},
            "payment_calendar_id": "nyse", "discount_curve_id": "USD-OIS",
            "pricing_overrides": {"mc_paths": 1}, "attributes": {},
            "prepayment_spec": prepay, "default_spec": default, "recovery_spec": recovery,
            "stochastic_prepay_spec": {"model": "Deterministic", **prepay},
            "stochastic_default_spec": {"model": "Deterministic", **default},
            "market_conditions": {"refi_rate": 0.04, "original_rate": None, "hpa": None,
                                  "unemployment": None, "seasonal_factor": 1.0, "custom_factors": {}},
            "credit_factors": {"credit_score": None, "dti": None, "ltv": None,
                               "delinquency_days": 0, "unemployment_rate": None, "custom_factors": {}}}
    if reinvestment_price is not None:
        spec["behavior_overrides"] = {"reinvestment_price": reinvestment_price}
    d = StructuredCredit(spec=spec)
    d.validate()
    return d, spec

## Market context

In [ ]:
# OIS discount curve (PV) + 3M-SOFR forward curve (projects floating coupons).
ois = DiscountCurve("USD-OIS", AS_OF_D,
                    [(0.0, 1.0), (1.0, 0.95), (3.0, 0.86), (5.0, 0.78), (8.0, 0.66), (10.0, 0.60)],
                    day_count="act_365f")
sofr = ForwardCurve("USD-SOFR-3M", 0.25,
                    [(0.0, 0.053), (3.0, 0.045), (8.0, 0.040)],
                    base_date=AS_OF_D, day_count="act_360")
market_json = MarketContext().insert(ois).insert(sofr).to_json()
print("Market:", [c.get("id") for c in json.loads(market_json)["curves"]])

In [ ]:
def tranche_results(deal):
    """Per-tranche Monte-Carlo results (npv, average_life, credit_duration, expected_loss)."""
    out = json.loads(deal.price(market_json, AS_OF, "structured_credit_stochastic"))
    return out["details"]["data"]["tranche_results"]

def price_table(deal, spec):
    """Tidy DataFrame of PV, price (% of par), WAL and expected loss per tranche."""
    orig = {t["id"]: int(t["original_balance"]["amount"]) for t in spec["tranches"]["tranches"]}
    rows = []
    for t in tranche_results(deal):
        ob = orig[t["tranche_id"]]
        pv = float(t["npv"]["amount"])
        rows.append({"tranche": t["tranche_id"], "seniority": t["seniority"],
                     "orig_$mm": ob / 1e6, "PV_$mm": pv / 1e6,
                     "price_%par": 100 * pv / ob if ob else float("nan"),
                     "WAL_yrs": t["average_life"], "credit_dur": t["credit_duration"],
                     "exp_loss_$mm": float(t["expected_loss"]["amount"]) / 1e6})
    return pd.DataFrame(rows)

## Auto ABS — senior note + residual

A \$6mm new/used auto pool, \$5.1mm senior **A** (fixed 3.5%) and \$0.9mm
**residual**, monthly pay, **130% PSA** prepayment, 2% CDR, 60% recovery.

In [ ]:
auto_assets = [
    asset("AUTO-NEW", {"type": "NewAutoLoan", "ltv": 0.85}, 3_500_000, 0.045, maturity="2029-06-01"),
    asset("AUTO-USED", {"type": "UsedAutoLoan", "ltv": 0.80}, 2_500_000, 0.055, maturity="2029-06-01"),
]
auto_tr = [
    tranche("A", 0.0, 85.0, "Senior", 5_100_000, fixed(0.035), 1, freq_n=1, maturity="2029-06-01"),
    tranche("R", 85.0, 100.0, "Equity", 900_000, fixed(0.0), 2, freq_n=1, maturity="2029-06-01"),
]
auto, auto_spec = build("AUTO-ABS", "ABS", pool("POOL-AUTO", "ABS", auto_assets),
                        auto_tr, 1, "2024-06-01", "2024-07-01", "2029-06-01",
                        prepay={"cpr": 0.06, "curve": {"curve": "psa", "speed_multiplier": 1.3}},
                        default={"cdr": 0.02, "curve": None},
                        recovery={"rate": 0.60, "recovery_lag": 3})
price_table(auto, auto_spec)

### Auto ABS stress (CDR × severity) and break-even CDR

In [ ]:
grid = json.dumps({"cprs": [0.0], "cdrs": [0.01, 0.03, 0.06, 0.10],
                   "severities": [0.40, 0.60], "recovery_lag": 3})
for tr in ("A", "R"):
    cells = json.loads(auto.scenario_table(market_json, AS_OF, tr, grid))["cells"]
    piv = pd.DataFrame(cells).pivot(index="cdr", columns="severity", values="price")
    print(f"Auto ABS tranche {tr} — price (% par); break-even CDR = "
          f"{auto.breakeven_cdr(market_json, AS_OF, tr) * 100:.2f}%")
    display(piv.style.format("{:.1f}"))

## RMBS — senior note + residual

A \$7mm single-family mortgage pool, \$5.6mm senior **A** (fixed 3.2%) and \$1.4mm
**residual**, monthly pay, **200% PSA** prepayment and a **100% SDA** default ramp.

In [ ]:
rmbs_assets = [
    asset("MTG-1", {"type": "SingleFamilyMortgage", "ltv": 0.75}, 4_000_000, 0.040, maturity="2054-01-01"),
    asset("MTG-2", {"type": "SingleFamilyMortgage", "ltv": 0.78}, 3_000_000, 0.042, maturity="2054-01-01"),
]
rmbs_tr = [
    tranche("A", 0.0, 80.0, "Senior", 5_600_000, fixed(0.032), 1, freq_n=1, maturity="2054-01-01"),
    tranche("R", 80.0, 100.0, "Equity", 1_400_000, fixed(0.0), 2, freq_n=1, maturity="2054-01-01"),
]
rmbs, rmbs_spec = build("RMBS", "RMBS", pool("POOL-RMBS", "RMBS", rmbs_assets),
                        rmbs_tr, 1, "2024-06-01", "2024-07-01", "2054-01-01",
                        prepay={"cpr": 0.0, "curve": {"curve": "psa", "speed_multiplier": 2.0}},
                        default={"cdr": 0.0, "curve": {"curve": "sda", "speed_multiplier": 1.0}},
                        recovery={"rate": 0.65, "recovery_lag": 6})
price_table(rmbs, rmbs_spec)

### RMBS prepayment sensitivity (PSA speed) and CPR × CDR stress

In [ ]:
# Senior WAL shortens sharply as PSA speed rises (classic prepayment risk).
rows = []
for speed in [1.0, 1.5, 2.0, 3.0]:
    d, s = build("RMBS-P", "RMBS", pool("P", "RMBS", rmbs_assets), rmbs_tr,
                 1, "2024-06-01", "2024-07-01", "2054-01-01",
                 prepay={"cpr": 0.0, "curve": {"curve": "psa", "speed_multiplier": speed}},
                 default={"cdr": 0.0, "curve": {"curve": "sda", "speed_multiplier": 1.0}},
                 recovery={"rate": 0.65, "recovery_lag": 6})
    a = next(t for t in tranche_results(d) if t["tranche_id"] == "A")
    rows.append({"PSA_%": int(speed * 100), "senior_WAL_yrs": a["average_life"],
                 "senior_PV_$mm": float(a["npv"]["amount"]) / 1e6})
display(pd.DataFrame(rows))

# scenario_table sweeps *constant* CPR (annual decimal, <= 1.0), not PSA multiples.
grid = json.dumps({"cprs": [0.10, 0.25], "cdrs": [0.005, 0.02, 0.05],
                   "severities": [0.35], "recovery_lag": 6})
cells = json.loads(rmbs.scenario_table(market_json, AS_OF, "R", grid))["cells"]
print("RMBS residual price (% par) over constant CPR × CDR:")
pd.DataFrame(cells).pivot(index="cdr", columns="cpr", values="price")

## Takeaways

- **Auto ABS / RMBS** are **amortizing, prepayment-driven** structures: the senior
  **WAL** is set by **PSA** speed, and the **residual** is first-loss to **CDR/SDA**.
- The senior debt absorbs default loss-free up to a healthy **break-even CDR**; the
  residual's price collapses as default and severity climb.
- Together with *structured credit 1–2*, this completes the tour: **CLO debt**,
  **CLO equity**, and **consumer ABS/RMBS** debt + residual — PV, WAL, spreads,
  break-even and full scenario grids, all from the same deal-spec API.